In [2]:
#Basic from scratch decision tree classifier implementation in Python


#Dependencies
import math
import numpy as np
import pandas as pd


In [52]:
#First we define the node class

class Node():

    def __init__(self,threshhold: float = None, Leaf: bool = False, ds_Y: list = [], index:int = 0, ds_X: list= []):

        self.threshold = threshhold
        self.Leaf = Leaf
        self.ds_y = ds_Y
        self.ds_X = ds_X
        self.index = index
        self.left = None
        self.right = None

        if Leaf == True:
            self.leaf_node()

    def evaluate(self, value):
        feature_val = value[self.index]

        if self.Leaf == True:
            self.leaf_node()
            return f'leaf node results: {self.class_dict}'
        
        if feature_val < self.threshold:
            return self.left.evaluate(value)
        
        else:
            return self.right.evaluate(value)
        
    def leaf_node(self):
        
        ds_y = self.ds_y
        class_dict: dict = {}
        total_instances: int = len(ds_y)

        for unique_class in set(ds_y):
            class_dict[unique_class] = 0
        

        for class_instance in ds_y:
            class_dict[class_instance] +=1
        

        for unique_class in class_dict:
            class_dict[unique_class] = class_dict[unique_class] / total_instances

        self.class_dict = class_dict

class DecisionTree():

    def __init__(self):

        self.model: list = []
        pass

    def gini(self, ds_Y: list) -> float:
        '''
        Gini = 1 - Sum((proportion of set)**2)
        '''
        gini:float = 0.0
        gini_dict: dict = {}
        no_elements: float = float(len(ds_Y))
        for value in ds_Y:
            if value not in gini_dict:
                gini_dict[value] = 0
            
            gini_dict[value] += 1.0
        
        for key in gini_dict:
            gini_dict[key] = (gini_dict[key]/no_elements)**2
            gini += gini_dict[key]

        return 1-gini

    def weighted_gini(self, ds_y_L, ds_y_R):

        gini_L = self.gini(ds_y_L)
        gini_R = self.gini(ds_y_R)

        proportion_left = len(ds_y_L)
        proportion_right = len(ds_y_R)
        total_proportion = proportion_left+proportion_right
        

        weighted_gini =(((gini_L)*(proportion_left/total_proportion))+((gini_R)*((proportion_right/total_proportion))))
        
        return weighted_gini

    def split_data(self, threshold, feature_index, ds_X, ds_y):
        ds_X_R = []
        ds_y_R = []

        ds_X_L = []
        ds_y_L = []

        for i in range(len(ds_X)):
            if ds_X[i][feature_index] > threshold:
                ds_X_R.append(ds_X[i])
                ds_y_R.append(ds_y[i])

            else:
                ds_X_L.append(ds_X[i])
                ds_y_L.append(ds_y[i])

        return ds_X_L, ds_y_L, ds_X_R, ds_y_R

    def stopping_condition(self):
        return False

    def split(self, ds_X, ds_y):
        
        best_gini: list = [1000000]
        for index, features in enumerate(ds_X):
                for feature_index, threshhold in enumerate(features):
                    ds_X_L, ds_y_L, ds_X_R, ds_y_R = self.split_data(threshhold, feature_index, ds_X, ds_y)

                    current_gini = self.weighted_gini(ds_y_L, ds_y_R)
                    current_ws = best_gini[0]

                    if current_gini<current_ws:
                        best_gini = [current_gini, threshhold, feature_index, ds_X_L, ds_X_R, ds_y_L, ds_y_R]

        return None if best_gini[0] == 1000000 else best_gini
    def fit(self, ds_X = None, ds_y = None, model = None, max_depth = 10):

        #[weighted_sum, threshhold, threshhold_index, ds_X_L, ds_X_R, ds_Y_L, ds_Y_R]

        max_depth -= 1
        if model == None:
            model = Node(ds_Y = ds_y , ds_X = ds_X)
            self.model = model
        if len(set(model.ds_y)) == 1:
            model.Leaf = True
            model.leaf_node()
            return
        if max_depth > 0: 
            split = self.split(model.ds_X, model.ds_y)

            if split == None:
                model.Leaf = True
                model.leaf_node()
                return

            model.threshold = split[1]

            model.index = split[2]
            if max_depth == 1:
            
                model.left = Node(ds_Y = split[5], ds_X = split[3], Leaf = True)

                model.right = Node(ds_Y = split[6], ds_X = split[4], Leaf = True)

                model.left.leaf_node()
                model.right.leaf_node()
            else:
                model.left = Node(ds_Y = split[5], ds_X = split[3])

                model.right = Node(ds_Y = split[6], ds_X = split[4])
            
            self.fit(model = model.left, max_depth=max_depth)
            #L and R
            self.fit(model = model.right, max_depth=max_depth)

    def predict(self, df_X):
        predictions: list = []
        for feature_value in df_X:
            predictions.append(self.model.evaluate(feature_value))
        
        return predictions

In [56]:

df_iris = pd.read_csv("iris.csv")

df_iris.drop('Id',axis=1,inplace=True)

df_iris_X = df_iris[['SepalLengthCm','SepalWidthCm','PetalLengthCm','PetalWidthCm']]

df_iris_y = df_iris.Species



flower_model = DecisionTree()

df_iris_X = df_iris_X.values.tolist()
df_iris_y = df_iris_y.values.tolist()


flower_model.fit(df_iris_X, df_iris_y)
test_list = [df_iris_X[100],df_iris_X[15],df_iris_X[25]]

print(df_iris_y[100], df_iris_y[15], df_iris_y[25])
predict = (flower_model.predict(test_list))

print(predict)
print(flower_model.model)


Iris-virginica Iris-setosa Iris-setosa
["leaf node results: {'Iris-virginica': 1.0}", "leaf node results: {'Iris-setosa': 1.0}", "leaf node results: {'Iris-setosa': 1.0}"]
